# Agent 2 – Synthèse Thématique (RAG)

## Rôle et Responsabilités

Cet agent est le **moteur de compréhension** du système multi-agents. Il combine la puissance de la **Retrieval-Augmented Generation (RAG)** avec les capacités de raisonnement de **Claude Sonnet** pour produire une synthèse structurée de la littérature scientifique.

### Pipeline de l'Agent

1. **Requête vectorielle** : Pour chaque thème de recherche, interroge ChromaDB pour récupérer les chunks les plus pertinents
2. **Construction du contexte RAG** : Assemble le contexte à partir des chunks récupérés
3. **Synthèse par LLM** : Envoie le contexte à  avec un prompt spécialisé
4. **Structuration JSON** : Le modèle retourne une synthèse structurée avec thèmes, méthodologies, résultats, etc.
5. **Persistance** : Sauvegarde les résultats en JSON pour l'Agent 3

### Thèmes analysés
- Méthodes de revue de littérature automatisée
- Détection de gaps de recherche
- NLP pour le texte scientifique
- Extraction de connaissances depuis des articles
- Systèmes multi-agents pour la recherche

### Modèle utilisé
`claude-sonnet-4-5` – modèle de haute qualité pour la synthèse complexe

In [1]:
# ============================================================
# Installation des dépendances
# ============================================================
# !pip install anthropic chromadb sentence-transformers python-dotenv

# ============================================================
# Imports
# ============================================================
import llm_adapter as anthropic  # adaptateur Gemini (gratuit)
import chromadb
from chromadb.utils import embedding_functions
from llm_adapter import GeminiEmbeddingFunction
import json
import os
import time
import re
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

print("Imports réussis.")
print(f"LLM adapter version : {anthropic.__version__}")
print(f"ChromaDB version      : {chromadb.__version__}")

Imports réussis.
LLM adapter version : llm_adapter/gemini-2.0-flash
ChromaDB version      : 1.5.9


In [2]:
# ============================================================
# Configuration et initialisation
# ============================================================

# Chargement des variables d'environnement (ANTHROPIC_API_KEY)
load_dotenv(dotenv_path=Path("../.env"))
ANTHROPIC_API_KEY = os.getenv("GEMINI_API_KEY", "")

if not ANTHROPIC_API_KEY:
    print("[ATTENTION] ANTHROPIC_API_KEY non trouvée dans .env")
    print("Créez un fichier .env avec : ANTHROPIC_API_KEY=votre_clé")
else:
    print(f"Clé API Anthropic chargée : {ANTHROPIC_API_KEY[:8]}...")

# Constantes de configuration
CHROMA_DB_PATH   = "../chroma_db"
COLLECTION_NAME  = "literature_review"
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"
SYNTHESIS_MODEL  = "claude-sonnet-4-5"   # Modèle puissant pour la synthèse
LOGS_DIR         = "../logs"
N_RESULTS        = 10                     # Chunks récupérés par requête
MAX_TOKENS       = 2048                   # Limite de tokens pour la réponse

# Initialiser le client Anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"Client Anthropic initialisé (modèle : {SYNTHESIS_MODEL})")

# Connexion à la collection ChromaDB existante (créée par l'Agent 1)
embedding_fn = GeminiEmbeddingFunction()
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

try:
    collection = chroma_client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    print(f"Collection '{COLLECTION_NAME}' connectée : {collection.count()} chunks disponibles")
except Exception as e:
    print(f"[ERREUR] Impossible de se connecter à la collection : {e}")
    print("Assurez-vous d'avoir exécuté l'Agent 1 (01_agent_ingestion.ipynb) d'abord.")
    collection = None

Clé API Anthropic chargée : AIza****...


Client Anthropic initialisé (modèle : claude-sonnet-4-5)


Collection 'literature_review' connectée : 258 chunks disponibles


In [3]:
# ============================================================
# Fonction de requête vers la base vectorielle
# ============================================================

def query_vector_db(query: str, n_results: int = 10) -> list:
    """Interroge ChromaDB et retourne les chunks les plus pertinents.
    
    Effectue une recherche par similarité vectorielle en utilisant
    l'embedding de la requête pour trouver les passages les plus
    sémantiquement proches dans la collection.
    
    Args:
        query: Requête textuelle en langage naturel.
        n_results: Nombre de chunks à récupérer (défaut: 10).
        
    Returns:
        Liste de dictionnaires contenant pour chaque chunk :
          - text (str): Contenu du chunk
          - metadata (dict): Métadonnées (fichier, page, index)
          - distance (float): Score de similarité (plus bas = plus proche)
    """
    if collection is None:
        print("[ERREUR] Collection ChromaDB non disponible")
        return []

    total_docs = collection.count()
    if total_docs == 0:
        print("[ATTENTION] La collection ChromaDB est vide")
        return []

    # Limiter n_results au nombre de documents disponibles
    n_results = min(n_results, total_docs)

    try:
        raw_results = collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )

        # Formater les résultats en liste de dicts lisibles
        chunks = []
        for doc, meta, dist in zip(
            raw_results["documents"][0],
            raw_results["metadatas"][0],
            raw_results["distances"][0]
        ):
            chunks.append({
                "text": doc,
                "metadata": meta,
                "distance": round(dist, 4)
            })

        return chunks

    except Exception as e:
        print(f"[ERREUR] Requête ChromaDB échouée : {e}")
        return []


print("Fonction query_vector_db définie.")

# Test de la fonction si la collection est disponible
if collection and collection.count() > 0:
    test_results = query_vector_db("literature review", n_results=2)
    print(f"Test : {len(test_results)} chunks récupérés pour 'literature review'")
else:
    print("Test ignoré : collection vide ou non disponible")

Fonction query_vector_db définie.


Test : 2 chunks récupérés pour 'literature review'


In [4]:
# ============================================================
# Prompt système de l'Agent de Synthèse
# ============================================================

SYNTHESIS_SYSTEM_PROMPT = """You are an expert scientific literature analyst specializing in systematic reviews and meta-analyses. Your role is to synthesize scientific papers and produce structured, actionable insights.

When given a set of excerpts from scientific papers related to a specific research theme, you must:

1. **Analyze key themes**: Identify the main topics, sub-topics, and recurring concepts across the papers
2. **Identify methodologies**: Extract and categorize the research methods used (experimental, theoretical, computational, survey-based, etc.)
3. **Synthesize findings**: Summarize the main results, conclusions, and contributions of the papers
4. **Track temporal evolution**: Identify how the field has evolved over time, noting shifts in focus or methodology
5. **Map key concepts and authors**: Identify foundational concepts and influential contributors

You MUST respond with a valid JSON object following this exact structure:
{
  "themes": [
    {
      "name": "Theme name",
      "description": "Brief description",
      "frequency": "high/medium/low",
      "related_papers": ["paper1.pdf", "paper2.pdf"]
    }
  ],
  "key_methodologies": [
    {
      "method": "Method name",
      "description": "How it is used in this domain",
      "papers_using_it": ["paper1.pdf"]
    }
  ],
  "main_findings": [
    {
      "finding": "Key finding statement",
      "significance": "Why this matters",
      "source_papers": ["paper1.pdf"]
    }
  ],
  "temporal_evolution": {
    "early_phase": "Description of early work",
    "middle_phase": "Description of middle period",
    "recent_phase": "Description of recent developments",
    "trend": "Overall trend description"
  },
  "key_authors_concepts": [
    {
      "concept": "Concept or author name",
      "importance": "Why this is central to the field",
      "references": ["paper1.pdf"]
    }
  ]
}

Rules:
- Base your analysis ONLY on the provided excerpts
- If information is insufficient for a field, write "Insufficient data from provided excerpts"
- Be precise and academic in tone
- Do NOT invent citations or papers not present in the context
- Return ONLY the JSON object, no additional text before or after"""

print("Prompt système SYNTHESIS_SYSTEM_PROMPT défini.")
print(f"Longueur du prompt : {len(SYNTHESIS_SYSTEM_PROMPT)} caractères")

Prompt système SYNTHESIS_SYSTEM_PROMPT défini.
Longueur du prompt : 2180 caractères


In [5]:
# ============================================================
# Fonction de synthèse thématique par RAG
# ============================================================

def synthesize_theme(theme_query: str, client: anthropic.Anthropic) -> dict:
    """Synthétise la littérature pour un thème donné via RAG + Claude.
    
    Pipeline :
    1. Requête vectorielle ChromaDB pour le thème
    2. Construction du contexte RAG à partir des chunks récupérés
    3. Appel à claude-sonnet-4-5 avec contexte + prompt système
    4. Parsing de la réponse JSON structurée
    
    Args:
        theme_query: Description du thème à analyser.
        client: Instance du client Anthropic.
        
    Returns:
        Dictionnaire de synthèse structurée ou dict minimal en cas d'erreur.
    """
    print(f"  Synthèse du thème : '{theme_query}'")

    # Étape 1 : Récupérer les chunks pertinents depuis ChromaDB
    relevant_chunks = query_vector_db(theme_query, n_results=N_RESULTS)

    if not relevant_chunks:
        print(f"  [ATTENTION] Aucun chunk trouvé pour le thème '{theme_query}'")
        return {
            "theme": theme_query,
            "status": "no_data",
            "themes": [],
            "key_methodologies": [],
            "main_findings": [],
            "temporal_evolution": {"trend": "Données insuffisantes"},
            "key_authors_concepts": []
        }

    # Étape 2 : Construire le contexte RAG
    context_parts = []
    for i, chunk in enumerate(relevant_chunks):
        source = chunk["metadata"].get("filename", "unknown")
        page = chunk["metadata"].get("total_pages", "?")
        chunk_idx = chunk["metadata"].get("chunk_index", i)
        context_parts.append(
            f"--- Extrait #{i+1} (Source: {source}, Chunk: {chunk_idx}, Pertinence: {1 - chunk['distance']:.3f}) ---\n"
            f"{chunk['text']}"
        )

    rag_context = "\n\n".join(context_parts)

    # Étape 3 : Construire le message utilisateur
    user_message = f"""Research Theme: {theme_query}

Below are {len(relevant_chunks)} excerpts from scientific papers retrieved from the literature database, ordered by relevance to the theme:

{rag_context}

Based on these excerpts, provide a comprehensive structured synthesis of this research theme following the JSON format specified in the system prompt."""

    try:
        # Étape 4 : Appel à l'API Claude
        print(f"    Contexte RAG : {len(relevant_chunks)} chunks | {len(rag_context)} caractères")
        print(f"    Appel à {SYNTHESIS_MODEL}...")

        response = client.messages.create(
            model=SYNTHESIS_MODEL,
            max_tokens=MAX_TOKENS,
            system=SYNTHESIS_SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": user_message}
            ]
        )

        raw_response = response.content[0].text.strip()

        # Étape 5 : Parser la réponse JSON
        # Nettoyer les balises markdown éventuelles
        if raw_response.startswith("```json"):
            raw_response = raw_response[7:]
        if raw_response.startswith("```"):
            raw_response = raw_response[3:]
        if raw_response.endswith("```"):
            raw_response = raw_response[:-3]

        synthesis = json.loads(raw_response.strip())

        # Enrichir le résultat avec des métadonnées
        synthesis["theme"] = theme_query
        synthesis["status"] = "success"
        synthesis["chunks_used"] = len(relevant_chunks)
        synthesis["model_used"] = SYNTHESIS_MODEL
        synthesis["input_tokens"] = response.usage.input_tokens
        synthesis["output_tokens"] = response.usage.output_tokens

        print(f"    Synthèse réussie : {response.usage.input_tokens} tokens in, {response.usage.output_tokens} tokens out")
        return synthesis

    except json.JSONDecodeError as je:
        # Le modèle n'a pas retourné du JSON valide
        print(f"    [ERREUR JSON] Parsing échoué : {je}")
        return {
            "theme": theme_query,
            "status": "json_parse_error",
            "raw_response": raw_response[:500] if 'raw_response' in locals() else "",
            "themes": [],
            "key_methodologies": [],
            "main_findings": [],
            "temporal_evolution": {"trend": "Erreur de parsing"},
            "key_authors_concepts": []
        }

    except anthropic.APIError as ae:
        # Erreur API Anthropic (rate limit, quota, etc.)
        print(f"    [ERREUR API] {ae}")
        return {
            "theme": theme_query,
            "status": "api_error",
            "error": str(ae),
            "themes": [],
            "key_methodologies": [],
            "main_findings": [],
            "temporal_evolution": {"trend": "Erreur API"},
            "key_authors_concepts": []
        }

    except Exception as e:
        # Fallback générique
        print(f"    [ERREUR] {e}")
        return {
            "theme": theme_query,
            "status": "error",
            "error": str(e),
            "themes": [],
            "key_methodologies": [],
            "main_findings": [],
            "temporal_evolution": {"trend": "Erreur inconnue"},
            "key_authors_concepts": []
        }


print("Fonction synthesize_theme définie.")

Fonction synthesize_theme définie.


In [6]:
# ============================================================
# Agent principal de Synthèse
# ============================================================

def run_synthesis_agent() -> dict:
    """Exécute l'agent de synthèse thématique complet.
    
    Itère sur tous les thèmes de recherche définis, synthétise
    la littérature pour chaque thème via RAG + Claude Sonnet,
    puis combine les résultats en une synthèse globale.
    
    Returns:
        Dictionnaire de synthèse globale contenant les résultats
        pour chaque thème et des statistiques d'usage.
    """
    start_time = time.time()
    print("=" * 60)
    print("AGENT 2 – SYNTHÈSE THÉMATIQUE (RAG)")
    print("=" * 60)
    print(f"Démarrage : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Modèle    : {SYNTHESIS_MODEL}")
    print()

    # Thèmes de recherche à analyser
    RESEARCH_THEMES = [
        "automated literature review methods",
        "research gap detection",
        "NLP for scientific text",
        "knowledge extraction from papers",
        "multi-agent systems for research"
    ]

    print(f"Thèmes à analyser : {len(RESEARCH_THEMES)}")
    for i, theme in enumerate(RESEARCH_THEMES):
        print(f"  {i+1}. {theme}")
    print()

    # Vérification préalable de la collection
    if collection is None or collection.count() == 0:
        print("[ATTENTION] Collection ChromaDB vide ou inaccessible.")
        print("L'agent va tenter de travailler mais les synthèses seront limitées.")

    # Synthèse par thème
    theme_syntheses = []
    total_input_tokens = 0
    total_output_tokens = 0

    print("Synthèse par thème :")
    print("-" * 40)

    for i, theme in enumerate(RESEARCH_THEMES):
        print(f"\n[{i+1}/{len(RESEARCH_THEMES)}] Thème : {theme}")
        synthesis = synthesize_theme(theme, client)
        theme_syntheses.append(synthesis)

        # Cumuler les tokens utilisés
        total_input_tokens += synthesis.get("input_tokens", 0)
        total_output_tokens += synthesis.get("output_tokens", 0)

        # Petite pause pour éviter le rate limiting
        if i < len(RESEARCH_THEMES) - 1:
            time.sleep(1)

    # Construire la synthèse globale
    duration = round(time.time() - start_time, 2)

    # Agréger toutes les méthodologies clés (déduplication par nom)
    all_methodologies = {}
    for synth in theme_syntheses:
        for meth in synth.get("key_methodologies", []):
            key = meth.get("method", "").lower()
            if key and key not in all_methodologies:
                all_methodologies[key] = meth

    # Agréger tous les thèmes identifiés
    all_themes = []
    for synth in theme_syntheses:
        for theme in synth.get("themes", []):
            all_themes.append({
                "research_query": synth.get("theme", ""),
                **theme
            })

    # Agréger les résultats principaux
    all_findings = []
    for synth in theme_syntheses:
        for finding in synth.get("main_findings", []):
            all_findings.append({
                "research_query": synth.get("theme", ""),
                **finding
            })

    global_synthesis = {
        "status": "completed",
        "model_used": SYNTHESIS_MODEL,
        "themes_analyzed": RESEARCH_THEMES,
        "theme_syntheses": theme_syntheses,
        "aggregated_themes": all_themes,
        "aggregated_methodologies": list(all_methodologies.values()),
        "aggregated_findings": all_findings,
        "usage_stats": {
            "total_themes": len(RESEARCH_THEMES),
            "successful_syntheses": sum(1 for s in theme_syntheses if s.get("status") == "success"),
            "failed_syntheses": sum(1 for s in theme_syntheses if s.get("status") != "success"),
            "total_input_tokens": total_input_tokens,
            "total_output_tokens": total_output_tokens,
            "duration_seconds": duration
        },
        "completed_at": datetime.now().isoformat()
    }

    # Sauvegarder les résultats en JSON
    logs_path = Path(LOGS_DIR)
    logs_path.mkdir(parents=True, exist_ok=True)
    output_path = logs_path / "synthesis_result.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(global_synthesis, f, indent=2, ensure_ascii=False)

    # Afficher le résumé
    print()
    print("=" * 60)
    print("RÉSUMÉ DE LA SYNTHÈSE")
    print("=" * 60)
    print(f"Thèmes analysés       : {len(RESEARCH_THEMES)}")
    print(f"Synthèses réussies    : {global_synthesis['usage_stats']['successful_syntheses']}")
    print(f"Synthèses échouées    : {global_synthesis['usage_stats']['failed_syntheses']}")
    print(f"Tokens utilisés       : {total_input_tokens} in + {total_output_tokens} out")
    print(f"Durée totale          : {duration}s")
    print(f"Résultats sauvegardés : {output_path}")

    return global_synthesis


print("Fonction run_synthesis_agent définie.")

Fonction run_synthesis_agent définie.


In [7]:
# ============================================================
# Exécution de l'Agent de Synthèse
# ============================================================

# Lancer l'agent
synthesis_results = run_synthesis_agent()

# Affichage structuré des résultats
print()
print("=" * 60)
print("APERÇU DES SYNTHÈSES PAR THÈME")
print("=" * 60)

for synth in synthesis_results.get("theme_syntheses", []):
    theme = synth.get("theme", "N/A")
    status = synth.get("status", "N/A")
    n_themes = len(synth.get("themes", []))
    n_methods = len(synth.get("key_methodologies", []))
    n_findings = len(synth.get("main_findings", []))

    print(f"\nThème : {theme}")
    print(f"  Statut        : {status}")
    print(f"  Sous-thèmes   : {n_themes}")
    print(f"  Méthodologies : {n_methods}")
    print(f"  Résultats     : {n_findings}")

    # Afficher les 2 premiers sous-thèmes identifiés
    for t in synth.get("themes", [])[:2]:
        print(f"    - {t.get('name', 'N/A')} [{t.get('frequency', 'N/A')}]")

    # Afficher la tendance temporelle
    temporal = synth.get("temporal_evolution", {})
    trend = temporal.get("trend", "N/A")
    if len(trend) > 100:
        trend = trend[:100] + "..."
    print(f"  Tendance      : {trend}")

print()
print("Résultats complets disponibles dans : ../logs/synthesis_result.json")

AGENT 2 – SYNTHÈSE THÉMATIQUE (RAG)
Démarrage : 2026-05-15 12:15:45
Modèle    : claude-sonnet-4-5

Thèmes à analyser : 5
  1. automated literature review methods
  2. research gap detection
  3. NLP for scientific text
  4. knowledge extraction from papers
  5. multi-agent systems for research

Synthèse par thème :
----------------------------------------

[1/5] Thème : automated literature review methods
  Synthèse du thème : 'automated literature review methods'


    Contexte RAG : 10 chunks | 8463 caractères
    Appel à claude-sonnet-4-5...


    [ERREUR] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 13.132890187s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go


[2/5] Thème : research gap detection
  Synthèse du thème : 'research gap detection'


    Contexte RAG : 10 chunks | 8561 caractères
    Appel à claude-sonnet-4-5...
    [ERREUR] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 11.533334776s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', '


[3/5] Thème : NLP for scientific text
  Synthèse du thème : 'NLP for scientific text'


    Contexte RAG : 10 chunks | 8437 caractères
    Appel à claude-sonnet-4-5...
    [ERREUR] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 9.914207078s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'u


[4/5] Thème : knowledge extraction from papers
  Synthèse du thème : 'knowledge extraction from papers'


    Contexte RAG : 10 chunks | 8452 caractères
    Appel à claude-sonnet-4-5...
    [ERREUR] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 8.31204518s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'ur


[5/5] Thème : multi-agent systems for research
  Synthèse du thème : 'multi-agent systems for research'


    Contexte RAG : 10 chunks | 8322 caractères
    Appel à claude-sonnet-4-5...
    [ERREUR] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 6.684943934s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'u

## Logs de Raisonnement – Agent 2

### Comment interpréter les résultats de synthèse

#### Structure de la sortie JSON

Chaque synthèse thématique contient :

| Champ | Description |
|-------|-------------|
| `themes` | Sous-thèmes identifiés avec fréquence (high/medium/low) |
| `key_methodologies` | Méthodes de recherche utilisées dans le corpus |
| `main_findings` | Résultats principaux extraits des articles |
| `temporal_evolution` | Évolution historique du champ de recherche |
| `key_authors_concepts` | Concepts fondateurs et auteurs clés |

#### Indicateurs de qualité

- **`status: success`** : Synthèse complète et JSON valide retourné par Claude
- **`status: no_data`** : Aucun chunk pertinent trouvé dans ChromaDB pour ce thème
- **`status: api_error`** : Erreur Anthropic (vérifier la clé API et le quota)
- **`status: json_parse_error`** : Claude a répondu mais pas en JSON valide (rare)

#### Interprétation de la pertinence RAG

La distance vectorielle est inversement proportionnelle à la pertinence :
- Distance **< 0.3** : Très pertinent (même sujet)
- Distance **0.3–0.6** : Pertinent (sujet connexe)
- Distance **> 0.6** : Peu pertinent (sujet éloigné)

#### Consommation de tokens

Le modèle `claude-sonnet-4-5` est utilisé pour sa capacité de raisonnement. Pour réduire les coûts, ajustez `N_RESULTS` (moins de chunks = moins de tokens) ou réduisez `MAX_TOKENS`.

#### Passage à l'Agent 3

Les résultats sont sauvegardés dans `../logs/synthesis_result.json`. L'Agent 3 (Détection de Gaps) chargera ce fichier pour analyser les lacunes de recherche à partir de la synthèse produite ici.